In [1]:
from ipywidgets import interact
import os

import cartopy.crs as ccrs
import cmcrameri.cm as ccm
from joblib import Parallel, delayed
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from tqdm import tqdm

import gplately

from lib.main import *

from parameters import parameters

In [2]:
# Plate model name
plate_model_name = parameters["plate_model_name"]

# Timespan for analysis
temporal_resolution = parameters["temporal_resolution"]
time_min = parameters["timespan"]["min"]
time_max = parameters["timespan"]["max"]
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

plate_model_dir = parameters["plate_model_dir"]
outputs_dir = parameters["outputs_dir"]
feat_maps_dir = parameters["feat_maps_dir"]

if not os.path.exists(outputs_dir):
    os.makedirs(outputs_dir, exist_ok=True)

feat_maps_dir = os.path.join(outputs_dir, feat_maps_dir)

subduction_data_filename = parameters["subduction_data_filename"]
subduction_data_filename = os.path.join(outputs_dir, subduction_data_filename)

nprocs = 8

In [3]:
plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

if os.path.isfile(subduction_data_filename):
    subduction_data = pd.read_csv(subduction_data_filename)
else:
    subduction_data = run_calculate_convergence(
        nprocs=nprocs,
        min_time=time_min,
        max_time=time_max,
        temporal_resolution=temporal_resolution,
        plate_reconstruction=plate_model,
        verbose=True,
    )
    
    subduction_data.to_csv(subduction_data_filename, index=False)

In [4]:
subduction_data_columns = subduction_data.columns.tolist()
features_plot = subduction_data_columns.copy()
features_plot.remove("lon")
features_plot.remove("lat")
features_plot.remove("age (Ma)")
features_plot.remove("subducting_plate_ID")
features_plot.remove("trench_plate_ID")

gplot = get_plot_topologies(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
    plate_reconstruction=plate_model,
    filter_topologies=True,
)

projection = ccrs.Mollweide(central_longitude=60)

In [5]:
@interact
def show_map(time=time_steps, feature=features_plot):
    gplot.time = time
    
    subduction_data_t = subduction_data[subduction_data["age (Ma)"] == time]

    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(projection=projection, facecolor="azure")

    gplot.plot_continents(ax, edgecolor="none", facecolor="tan", alpha=0.5, zorder=1)
    gplot.plot_coastlines(ax, edgecolor="none", facecolor="tan", alpha=0.7, zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)
    
    feat = ax.scatter(subduction_data_t["lon"], subduction_data_t["lat"], 50, marker=".",
                      c=subduction_data_t[feature], cmap=ccm.hawaii_r, transform=ccrs.PlateCarree(), zorder=4)

    gplot.plot_all_topologies(ax, color="dimgray", linewidth=1.5, zorder=5)
    gplot.plot_trenches(ax, color="k", alpha=0.3, zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color="k", alpha=0.3, zorder=7)
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=8)
    
    ax.text(0.49,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, "60°W", transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
        
    cbar_feat = fig.colorbar(feat, shrink=0.4, pad=0.06, orientation="horizontal", extend="both")
    cbar_feat.set_label(feature, fontsize=16, labelpad=10)
    cbar_feat.ax.tick_params(labelsize=16)
    
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor="tan", edgecolor="none", label="Continental Crust"),
        Line2D([0], [0], color="dimgray", lw=2, label="Mid-Ocean Ridge")
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, fontsize=16, loc="lower left", bbox_to_anchor=(0, -0.2))
    
    ax.set_title(f"{time} Ma", fontsize=25, y=1.04)
        
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, …

In [6]:
if os.path.exists(feat_maps_dir):
    print(f"Feature maps are located in {feat_maps_dir}")
else:
    os.makedirs(feat_maps_dir, exist_ok=True)
    
    generate_feat_maps(
        gplot,
        subduction_data,
        projection,
        time_steps,
        feature="convergence_rate (cm/yr)",
        output_dir=feat_maps_dir,
        num_jobs=nprocs
    )
    
    output_filenames = [
        os.path.join(feat_maps_dir, f"feat_map_{t:0.0f}Ma.png")
        for t in time_steps
    ]
    
    output_filename = os.path.join(outputs_dir, "feat_animation.mp4")
    create_animation(
        image_filenames=output_filenames[::-1],
        output_filename=output_filename,
        fps=10,
        bitrate="5000k",
    )

Dispatching tasks: 100%|█████████████████████████████████████████████████████████████| 361/361 [14:34<00:00,  2.42s/it]
